In [1]:
import pandas as pd

# Load TrustMesh economic data
df = pd.read_csv("../data/trustmesh_economic_data.csv")
df["month"] = pd.to_datetime(df["month"])

# Use the latest economic state for each business
profiles = (
    df.sort_values("month")
      .groupby("business_id")
      .tail(1)
      .reset_index(drop=True)
)

print("Business profiles:", profiles.shape)
print("Businesses:", profiles["business_id"].nunique())

Business profiles: (500, 32)
Businesses: 500


In [2]:
def explain_business(row):
    reasons = []

    if row["cash_flow_margin_pct"] < 20:
        reasons.append("Low cash-flow margin")

    if row["expense_burden_pct"] > 60:
        reasons.append("High expense burden")

    if row["payment_discipline_pct"] < 70:
        reasons.append("Weak payment discipline")

    if row["supplier_reliability_pct"] < 70:
        reasons.append("Low supplier reliability")

    if row["demand_stability_pct"] < 70:
        reasons.append("Unstable demand")

    if row["business_continuity_pct"] < 70:
        reasons.append("Lower business continuity")

    if row["resilience_pct"] < 70:
        reasons.append("Lower economic resilience")

    if not reasons:
        reasons.append("No major weaknesses identified")

    return reasons


profiles["explanation"] = profiles.apply(explain_business, axis=1)

profiles["explanation_count"] = profiles["explanation"].str.len()

profiles["explanation"] = profiles["explanation"].apply(
    lambda x: "; ".join(x)
)

print("Businesses explained:", len(profiles))
print(
    "Average explanation factors:",
    profiles["explanation_count"].mean().round(2)
)

print("\nSample explanations:")
print(
    profiles[
        [
            "business_id",
            "economic_reputation_index",
            "explanation_count",
            "explanation"
        ]
    ].head(10).to_string(index=False)
)

Businesses explained: 500
Average explanation factors: 1.55

Sample explanations:
business_id  economic_reputation_index  explanation_count                                                                         explanation
       B490                      72.30                  1                                                      No major weaknesses identified
       B481                      55.48                  4 Low cash-flow margin; High expense burden; Weak payment discipline; Unstable demand
       B486                      61.81                  2                                           Low cash-flow margin; High expense burden
       B497                      67.35                  1                                                      No major weaknesses identified
       B493                      66.40                  2                                           Low supplier reliability; Unstable demand
       B495                      63.82                  1         

In [3]:
def interpret_score(score):
    if score >= 75:
        return "Strong"
    elif score >= 60:
        return "Moderate"
    elif score >= 50:
        return "Weak"
    return "Critical"


profiles["reputation_status"] = profiles["economic_reputation_index"].apply(
    interpret_score
)

print("Reputation status:")
print(profiles["reputation_status"].value_counts())

print("\nSample explainable results:")
print(
    profiles[
        [
            "business_id",
            "economic_reputation_index",
            "reputation_status",
            "explanation"
        ]
    ].head(10).to_string(index=False)
)

Reputation status:
reputation_status
Moderate    412
Strong       48
Weak         40
Name: count, dtype: int64

Sample explainable results:
business_id  economic_reputation_index reputation_status                                                                         explanation
       B490                      72.30          Moderate                                                      No major weaknesses identified
       B481                      55.48              Weak Low cash-flow margin; High expense burden; Weak payment discipline; Unstable demand
       B486                      61.81          Moderate                                           Low cash-flow margin; High expense burden
       B497                      67.35          Moderate                                                      No major weaknesses identified
       B493                      66.40          Moderate                                           Low supplier reliability; Unstable demand
       B495   

In [4]:
explainability_output = profiles[
    [
        "business_id",
        "owner_id",
        "business_type",
        "environment_type",
        "location",
        "economic_reputation_index",
        "reputation_status",
        "explanation_count",
        "explanation"
    ]
].copy()

explainability_output.to_csv(
    "../data/business_explanations.csv",
    index=False
)

print("Explainability output saved.")
print("Shape:", explainability_output.shape)

print("\nReputation status:")
print(explainability_output["reputation_status"].value_counts())

print("\nOutput preview:")
print(explainability_output.head(10).to_string(index=False))

Explainability output saved.
Shape: (500, 9)

Reputation status:
reputation_status
Moderate    412
Strong       48
Weak         40
Name: count, dtype: int64

Output preview:
business_id owner_id          business_type environment_type  location  economic_reputation_index reputation_status  explanation_count                                                                         explanation
       B490     O490 Micro Service Business       Semi-Urban Ahmedabad                      72.30          Moderate                  1                                                      No major weaknesses identified
       B481     O481            Food Vendor       Semi-Urban    Jaipur                      55.48              Weak                  4 Low cash-flow margin; High expense burden; Weak payment discipline; Unstable demand
       B486     O486            Retail Shop            Urban Ahmedabad                      61.81          Moderate                  2                                   

In [8]:
factor_counts = {
    "Low cash-flow margin": (profiles["cash_flow_margin_pct"] < 20).sum(),
    "High expense burden": (profiles["expense_burden_pct"] > 60).sum(),
    "Weak payment discipline": (profiles["payment_discipline_pct"] < 70).sum(),
    "Low supplier reliability": (profiles["supplier_reliability_pct"] < 70).sum(),
    "Unstable demand": (profiles["demand_stability_pct"] < 70).sum(),
    "Lower business continuity": (profiles["business_continuity_pct"] < 70).sum(),
    "Lower economic resilience": (profiles["resilience_pct"] < 70).sum()
}

factor_summary = (
    pd.Series(factor_counts)
    .sort_values(ascending=False)
    .rename("business_count")
    .reset_index()
    .rename(columns={"index": "factor"})
)

print("Most common explanation factors:")
print(factor_summary.to_string(index=False))

Most common explanation factors:
                   factor  business_count
          Unstable demand             152
     Low cash-flow margin             140
 Low supplier reliability             119
      High expense burden             102
  Weak payment discipline              52
Lower economic resilience              22
Lower business continuity              13


In [6]:
print("Cash-flow margin:")
print(profiles["cash_flow_margin_pct"].describe())

print("\nExpense burden:")
print(profiles["expense_burden_pct"].describe())

Cash-flow margin:
count    500.000000
mean      26.182680
std       13.977497
min      -18.730000
25%       18.662500
50%       25.905000
75%       34.587500
max       59.800000
Name: cash_flow_margin_pct, dtype: float64

Expense burden:
count    500.000000
mean      52.848580
std       10.535238
min       29.970000
25%       45.297500
50%       53.325000
75%       59.205000
max       83.280000
Name: expense_burden_pct, dtype: float64


In [7]:
print("Cash-flow margin below 20:",
      (profiles["cash_flow_margin_pct"] < 20).sum())

print("Expense burden above 60:",
      (profiles["expense_burden_pct"] > 60).sum())

print("Expense burden sample:")
print(
    profiles["expense_burden_pct"]
    .sort_values(ascending=False)
    .head(10)
    .to_list()
)

Cash-flow margin below 20: 140
Expense burden above 60: 102
Expense burden sample:
[83.28, 82.93, 82.49, 78.98, 78.43, 78.27, 78.26, 77.5, 76.62, 76.58]


In [9]:
factor_summary.to_csv(
    "../data/explanation_factor_summary.csv",
    index=False
)

print("Factor summary saved.")
print("Shape:", factor_summary.shape)

Factor summary saved.
Shape: (7, 2)


## Explainable AI Findings

The Explainable AI layer provides an interpretable explanation for each business's Economic Reputation Index.

Instead of presenting only a reputation score, the system identifies the underlying economic factors that may require attention, including cash-flow margin, expense burden, payment discipline, supplier reliability, demand stability, business continuity, and economic resilience.

Each business is also assigned a reputation status based on its Economic Reputation Index.

The current results contain **500 explainable business profiles**:

* **48 Strong**
* **412 Moderate**
* **40 Weak**
* **0 Critical**

The most common explanation factors identified across the business profiles were:

* **152** businesses with unstable demand
* **140** with low cash-flow margin
* **119** with low supplier reliability
* **102** with high expense burden
* **52** with weak payment discipline
* **22** with lower economic resilience
* **13** with lower business continuity

These factors are not mutually exclusive, as a business can have multiple economic weaknesses contributing to its explanation.

The explainability outputs are saved as:

* `data/business_explanations.csv`
* `data/explanation_factor_summary.csv`

This layer improves transparency by connecting reputation scores to specific economic indicators and provides an interpretable foundation for the future TrustMesh AI dashboard and decision-support system.
